In [6]:
import pandas as pd
import os

# 1. MAPEAMENTO E CONFIGURAÇÃO DE CAMINHOS
caminho_raw = '/home/bruno/UESC/DataSUS/DadosTUB/data/raw/'
caminho_salvar = '/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades.csv'

mapa_cidades = {
    '291360': 'Ilhéus', '291480': 'Itabuna',
    '291950': 'Vitória da Conquista', '291080': 'Feira de Santana', '292740': 'Salvador',
    '261160': 'Recife', '280030': 'Aracaju', '330455': 'Rio de Janeiro',
    '355030': 'São Paulo', '410690': 'Curitiba'
}
codigos_alvo = list(mapa_cidades.keys())
lista_dfs = []

# 2. LOOP DE EXTRAÇÃO INTELIGENTE (2014 a 2024)
print("Iniciando a extração com detecção automática de separadores...")

for ano in range(14, 25):
    arquivo_nome = f'TUBEBR{ano}.csv'
    caminho_arquivo = os.path.join(caminho_raw, arquivo_nome)
    
    if not os.path.exists(caminho_arquivo):
        arquivo_nome = f'TUBBR{ano}.csv'
        caminho_arquivo = os.path.join(caminho_raw, arquivo_nome)

    if os.path.exists(caminho_arquivo):
        print(f"Lendo: {arquivo_nome}...", end="")
        
        # --- NOVIDADE: FAREJADOR DE PADRÃO ---
        # Tenta descobrir o encoding e o separador olhando só a 1ª linha
        try:
            with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                primeira_linha = f.readline()
            encoding_usado = 'utf-8'
        except UnicodeDecodeError:
            with open(caminho_arquivo, 'r', encoding='latin-1') as f:
                primeira_linha = f.readline()
            encoding_usado = 'latin-1'
            
        separador_usado = ';' if ';' in primeira_linha else ','
        
        # --- LEITURA PELO PANDAS COM O PADRÃO CORRETO ---
        try:
            # error_bad_lines=False (ou on_bad_lines='skip') garante que se houver uma única linha 
            # com erro de digitação bizarra, ele ignora ela e salva o resto do ano
            df_ano = pd.read_csv(caminho_arquivo, sep=separador_usado, encoding=encoding_usado, 
                                 low_memory=False, on_bad_lines='skip')
        except Exception as e:
            print(f" Erro crítico do Pandas: {e}")
            continue
            
        # Detecta a coluna de notificação
        col_municipio = None
        for col_teste in ['ID_MUNICIP', 'Municipio_Notificacao']:
            if col_teste in df_ano.columns:
                col_municipio = col_teste
                break
        
        if col_municipio:
            # Formata os códigos de município
            df_ano[col_municipio] = df_ano[col_municipio].astype(str).str.replace('.0', '', regex=False).str[:6]
            
            # Filtra apenas nossas 10 cidades alvo
            df_filtrado = df_ano[df_ano[col_municipio].isin(codigos_alvo)].copy()
            
            lista_dfs.append(df_filtrado)
            print(f" -> {len(df_filtrado)} casos. (Sep: '{separador_usado}', Enc: {encoding_usado})")
        else:
            print(f" -> AVISO: Coluna de município não encontrada.")
    else:
        print(f"Arquivo não encontrado: {caminho_arquivo}")

# 3. CONSOLIDAÇÃO E SALVAMENTO
if lista_dfs:
    print("\nConcatenando os dados da década inteira...")
    df_final = pd.concat(lista_dfs, ignore_index=True)
    
    coluna_referencia = 'ID_MUNICIP' if 'ID_MUNICIP' in df_final.columns else 'Municipio_Notificacao'
    df_final['Cidade'] = df_final[coluna_referencia].map(mapa_cidades)
    
    print(f"Extração 100% concluída! Total de registros unificados: {len(df_final)}")
    
    df_final.to_csv(caminho_salvar, index=False)
    print(f"Novo arquivo salvo e pronto para a pirâmide etária!")
else:
    print("\nNenhum dado extraído.")

Iniciando a extração com detecção automática de separadores...
Lendo: TUBEBR14.csv... -> 21346 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR15.csv... -> 21555 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR16.csv... -> 21790 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR17.csv... -> 21982 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR18.csv... -> 23197 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR19.csv... -> 22773 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR20.csv... -> 20597 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR21.csv... -> 22019 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR22.csv... -> 24238 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR23.csv... -> 25692 casos. (Sep: ',', Enc: utf-8)
Lendo: TUBEBR24.csv... -> 25823 casos. (Sep: ',', Enc: utf-8)

Concatenando os dados da década inteira...
Extração 100% concluída! Total de registros unificados: 251012
Novo arquivo salvo e pronto para a pirâmide etária!


## Qualidade dos dados

In [9]:
import pandas as pd
import numpy as np
import json

# ==============================================================================
# 1. O DICIONÁRIO DE DADOS COMPLETO (SINAN TUBERCULOSE)
# ==============================================================================
dicionario_sinan_completo = {
    'TP_NOT': 'Tipo_Notificacao', 'ID_AGRAVO': 'Agravo_Codigo', 'DT_NOTIFIC': 'Data_Notificacao', 
    'NU_ANO': 'Ano_Notificacao', 'SG_UF_NOT': 'UF_Notificacao', 'ID_MUNICIP': 'Municipio_Notificacao', 
    'ID_REGIONA': 'Regional_Notificacao', 'TPUNINOT': 'Tipo_Unidade_Notificacao', 
    'NDUPLIC_N': 'Indicador_Duplicidade_Sistema', 'IN_VINCULA': 'Indicador_Vinculo_Sistema', 
    'MIGRADO_W': 'Indicador_Migrado_Web', 'DT_DIGITA': 'Data_Digitacao_Sistema', 
    'DT_TRANSUS': 'Data_Envio_SUS', 'DT_TRANSDM': 'Data_Envio_Distrito', 
    'DT_TRANSSM': 'Data_Envio_Municipio', 'DT_TRANSRM': 'Data_Envio_Regional', 
    'DT_TRANSRS': 'Data_Envio_Estado', 'DT_TRANSSE': 'Data_Envio_Secretaria_Estadual', 
    'CS_FLXRET': 'Indicador_Fluxo_Retorno', 'FLXRECEBI': 'Indicador_Fluxo_Recebimento', 
    'DT_DIAG': 'Data_Diagnostico', 'ANO_NASC': 'Ano_Nascimento', 'NU_IDADE_N': 'Idade_Codigo', 
    'CS_SEXO': 'Sexo', 'CS_GESTANT': 'Gestante', 'CS_RACA': 'Raca_Cor', 'CS_ESCOL_N': 'Escolaridade', 
    'ID_OCUPA_N': 'Ocupacao', 'SG_UF': 'UF_Residencia', 'ID_MN_RESI': 'Municipio_Residencia', 
    'ID_RG_RESI': 'Regional_Residencia', 'ID_PAIS': 'Pais_Residencia', 'POP_LIBER': 'Populacao_Privada_Liberdade', 
    'POP_RUA': 'Populacao_Situacao_Rua', 'POP_SAUDE': 'Profissional_Saude', 'POP_IMIG': 'Imigrante', 
    'BENEF_GOV': 'Beneficiario_Programa_Governo', 'INSTITUCIO': 'Paciente_Institucionalizado', 
    'TRATAMENTO': 'Tipo_Entrada', 'FORMA': 'Forma_Clinica', 'EXTRAPU1_N': 'Local_Extrapulmonar_1', 
    'EXTRAPU2_N': 'Local_Extrapulmonar_2', 'AGRAVAIDS': 'Agravo_AIDS', 'AGRAVALCOO': 'Agravo_Alcoolismo', 
    'AGRAVDIABE': 'Agravo_Diabetes', 'AGRAVDOENC': 'Agravo_Doenca_Mental', 'AGRAVTABAC': 'Agravo_Tabagismo', 
    'AGRAVDROGA': 'Agravo_Drogas_Ilicitas', 'AGRAVOUTRA': 'Agravo_Outras_Doencas', 'HIV': 'Sorologia_HIV', 
    'ANT_RETRO': 'Uso_Antiretroviral', 'DOENCA_TRA': 'Relacionado_Doenca_Trabalho', 'RAIOX_TORA': 'RaioX_Torax', 
    'TESTE_TUBE': 'Teste_Tuberculinico', 'TEST_MOLEC': 'Teste_Rapido_Molecular', 'TEST_SENSI': 'Teste_Sensibilidade', 
    'HISTOPATOL': 'Exame_Histopatologico', 'BACILOSC_E': 'Baciloscopia_Escarro_Diagnostico', 'CULTURA_ES': 'Cultura_Escarro', 
    'CULTURA_OU': 'Cultura_Outro_Material', 'BACILOS_E2': 'Baciloscopia_Escarro_2_Diagnostico', 
    'BACILOSC_O': 'Baciloscopia_Outros_Materiais', 'BACILOSC_1': 'Baciloscopia_1_Mes', 'BACILOSC_2': 'Baciloscopia_2_Mes', 
    'BACILOSC_3': 'Baciloscopia_3_Mes', 'BACILOSC_4': 'Baciloscopia_4_Mes', 'BACILOSC_5': 'Baciloscopia_5_Mes', 
    'BACILOSC_6': 'Baciloscopia_6_Mes', 'BAC_APOS_6': 'Baciloscopia_Apos_6_Meses', 'DT_INIC_TR': 'Data_Inicio_Tratamento', 
    'RIFAMPICIN': 'Droga_Rifampicina', 'ISONIAZIDA': 'Droga_Isoniazida', 'ETAMBUTOL': 'Droga_Etambutol', 
    'ESTREPTOMI': 'Droga_Estreptomicina', 'PIRAZINAMI': 'Droga_Pirazinamida', 'ETIONAMIDA': 'Droga_Etionamida', 
    'OUTRAS': 'Drogas_Outras', 'DT_MUDANCA': 'Data_Mudanca_Esquema', 'TRAT_SUPER': 'Tratamento_Supervisionado_Realizado', 
    'SG_UF_AT': 'UF_Atual_Tratamento', 'ID_MUNIC_A': 'Municipio_Atual_Tratamento', 'DT_NOTI_AT': 'Data_Atualizacao_Notificacao', 
    'TRANSF': 'Transferencia_Confirmada', 'UF_TRANSF': 'UF_Transferencia', 'MUN_TRANSF': 'Municipio_Transferencia', 
    'SG_UF_2': 'UF_2_Transferencia', 'ID_MUNIC_2': 'Municipio_2_Transferencia', 'TRATSUP_AT': 'Tratamento_Supervisionado_Atual', 
    'NU_CONTATO': 'Numero_Contatos_Identificados', 'NU_COMU_EX': 'Numero_Contatos_Examinados', 
    'SITUA_9_M': 'Situacao_9_Meses', 'SITUA_12_M': 'Situacao_12_Meses', 'SITUA_ENCE': 'Situacao_Encerramento_Final', 
    'DT_ENCERRA': 'Data_Encerramento'
}

# ==============================================================================
# 2. CARREGAMENTO E LIMPEZA INICIAL
# ==============================================================================
caminho_base = '/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades.csv'

# Lemos a base toda como string (dtype=str) para evitar inferências erradas do Pandas
df_regiao = pd.read_csv(caminho_base, sep=',', low_memory=False, dtype=str)

df_regiao.columns = df_regiao.columns.str.strip()
df_regiao.rename(columns=dicionario_sinan_completo, inplace=True, errors='ignore')

print(f"Total de registros carregados: {len(df_regiao)} notificações.\n")

# Tratamento de Nulos e Completude
# O bloco "future.no_silent_downcasting" desativa o aviso do Pandas nas versões novas
with pd.option_context('future.no_silent_downcasting', True):
    df_regiao = df_regiao.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    df_regiao.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    df_regiao.replace(['NaN', 'nan', 'NaM', 'None', ''], np.nan, inplace=True)

total_registros = len(df_regiao)
nulos_absolutos = df_regiao.isnull().sum()
perc_nulos = (nulos_absolutos / total_registros) * 100

# ==============================================================================
# 3. FILTRAGEM (Regra dos 45% + Exceções)
# ==============================================================================
excecoes_conhecidas = [
    'Baciloscopia_1_Mes', 'Baciloscopia_2_Mes', 'Baciloscopia_3_Mes',
    'Baciloscopia_4_Mes', 'Baciloscopia_5_Mes', 'Baciloscopia_6_Mes',
    'Baciloscopia_Apos_6_Meses', 'Situacao_9_Meses', 'Situacao_12_Meses',
    'Transferencia_Confirmada', 'UF_Transferencia', 'Municipio_Transferencia',
    'Data_Mudanca_Esquema', 'Local_Extrapulmonar_2', 'EXTRAPUL_O',
    'AGRAVOUTDE', 'OUTRAS_DES',
    'Baciloscopia_Outros_Materiais', 'Cultura_Outro_Material', 'Baciloscopia_Escarro_2_Diagnostico'
]

# Identifica as colunas com >= 45% de nulos
colunas_criticas = perc_nulos[perc_nulos >= 45.0].index.tolist()

# Separa para deletar apenas as que não estão na lista de exceções
colunas_para_deletar = [col for col in colunas_criticas if col not in excecoes_conhecidas]

df_limpo = df_regiao.drop(columns=colunas_para_deletar).copy()

print(f" Variáveis descartadas (>45% nulos): {len(colunas_para_deletar)}")
print(f" Variáveis mantidas no dataset final: {len(df_limpo.columns)}\n")

# ==============================================================================
# 4. TABELA VISUAL DE RESUMO SÊNIOR
# ==============================================================================
colunas_aprovadas = df_limpo.columns.tolist()

df_resumo_aprovadas = pd.DataFrame({
    'Variáveis Aprovadas (No Dataset Final)': colunas_aprovadas,
    'Dados Ausentes (%)': perc_nulos[colunas_aprovadas].values
})

df_resumo_aprovadas['Motivo da Manutenção'] = df_resumo_aprovadas['Variáveis Aprovadas (No Dataset Final)'].apply(
    lambda x: 'Exceção Clínica Valiosa' if x in excecoes_conhecidas else 'Preenchimento Adequado (<45%)'
)

df_resumo_aprovadas.sort_values(by=['Motivo da Manutenção', 'Dados Ausentes (%)'], ascending=[False, False], inplace=True)
df_resumo_aprovadas.reset_index(drop=True, inplace=True)

tabela_visual_aprovadas = df_resumo_aprovadas.style.format({
    'Dados Ausentes (%)': '{:.2f}%'
}).map(
    lambda val: 'color: blue; font-weight: bold' if 'Exceção' in val else 'color: green',
    subset=['Motivo da Manutenção']
)

display(tabela_visual_aprovadas)

# ==============================================================================
# 5. SALVAMENTO DOS ARQUIVOS LIMPOS
# ==============================================================================
nome_arquivo_dados = '/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades_limpo_2014_2024.csv'
df_limpo.to_csv(nome_arquivo_dados, index=False, encoding='utf-8')

nome_arquivo_json = '/home/bruno/UESC/DataSUS/DadosTUB/src/dicionario_sinan_tuberculose.json'
with open(nome_arquivo_json, 'w', encoding='utf-8') as f:
    json.dump(dicionario_sinan_completo, f, ensure_ascii=False, indent=4)

print(f"\nArquivos salvos com sucesso em:\n- {nome_arquivo_dados}\n- {nome_arquivo_json}")

Total de registros carregados: 251012 notificações.

 Variáveis descartadas (>45% nulos): 26
 Variáveis mantidas no dataset final: 72



,Variáveis Aprovadas (No Dataset Final),Dados Ausentes (%),Motivo da Manutenção
0,Tratamento_Supervisionado_Realizado,42.34%,Preenchimento Adequado (<45%)
1,Data_Atualizacao_Notificacao,35.63%,Preenchimento Adequado (<45%)
2,Tipo_Unidade_Notificacao,29.22%,Preenchimento Adequado (<45%)
3,Agravo_Outras_Doencas,26.18%,Preenchimento Adequado (<45%)
4,Beneficiario_Programa_Governo,24.22%,Preenchimento Adequado (<45%)
5,Municipio_2_Transferencia,23.50%,Preenchimento Adequado (<45%)
6,UF_2_Transferencia,23.41%,Preenchimento Adequado (<45%)
7,Pais_Residencia,23.33%,Preenchimento Adequado (<45%)
8,Teste_Sensibilidade,17.44%,Preenchimento Adequado (<45%)
9,Tratamento_Supervisionado_Atual,14.54%,Preenchimento Adequado (<45%)



Arquivos salvos com sucesso em:
- /home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades_limpo_2014_2024.csv
- /home/bruno/UESC/DataSUS/DadosTUB/src/dicionario_sinan_tuberculose.json


## Formatação dos dados

In [10]:
import json
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS LIMPOS
# ==============================================================================
try:
    with open('/home/bruno/UESC/DataSUS/DadosTUB/src/dicionario_sinan_tuberculose.json', 'r', encoding='utf-8') as f:
        dicionario = json.load(f)
except FileNotFoundError:
    print("Dicionário JSON não encontrado. O script seguirá com os nomes atuais das colunas.")

caminho_base_limpa = '/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades_limpo_2014_2024.csv'
df = pd.read_csv(caminho_base_limpa, low_memory=False, dtype=str)

# ==============================================================================
# 2. CONVERSÃO DE DATAS E ENGENHARIA DE IDADE
# ==============================================================================
colunas_de_data = [col for col in df.columns if 'Data' in col]
for col in colunas_de_data:
    # errors='coerce' transformará datas impossíveis do SUS (ex: 2014-02-30) em NaT
    df[col] = pd.to_datetime(df[col], errors='coerce') 

def descodificar_idade(codigo):
    if pd.isna(codigo):
        return np.nan
    
    codigo = str(codigo).strip().replace('.0', '')
    if len(codigo) != 4:
        return np.nan
        
    tipo = codigo[0]
    valor = int(codigo[1:])
    
    if tipo == '4':   # 4 = Idade em Anos
        return valor
    elif tipo == '3': # 3 = Idade em Meses (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '2': # 2 = Idade em Dias (Menor de 1 ano = 0 anos)
        return 0
    elif tipo == '5': # 5 = Idade acima de 100 anos (valor já é em anos)
        return valor
    else:
        return np.nan
        
if 'Idade_Codigo' in df.columns:
    df['Idade_Anos'] = df['Idade_Codigo'].apply(descodificar_idade)
    df.drop(columns=['Idade_Codigo'], inplace=True)
elif 'NU_IDADE_N' in df.columns: # Caso a coluna ainda esteja com o nome bruto
    df['Idade_Anos'] = df['NU_IDADE_N'].apply(descodificar_idade)

# ==============================================================================
# 3. MAPEAMENTO DE CATEGORIAS (DE-PARA)
# ==============================================================================
mapa_sexo = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'}

mapa_encerramento = {
    '1': 'Cura', '2': 'Abandono', '3': 'Óbito por TB', 
    '4': 'Óbito por outras causas', '5': 'Transferência', 
    '6': 'Mudança de Diagnóstico', '7': 'TB-DR', 
    '8': 'Mudança de Esquema', '9': 'Falência', '10': 'Abandono Primário'
}

mapa_entrada = {
    '1': 'Caso Novo', '2': 'Recidiva', '3': 'Reingresso após abandono', 
    '4': 'Não sabe', '5': 'Transferência', '6': 'Pós-óbito'
}

# Desativando o FutureWarning do Pandas para as próximas linhas
with pd.option_context('future.no_silent_downcasting', True):
    if 'Sexo' in df.columns:
        df['Sexo'] = df['Sexo'].map(mapa_sexo).fillna(df['Sexo'])
    elif 'CS_SEXO' in df.columns:
        df['Sexo'] = df['CS_SEXO'].map(mapa_sexo).fillna(df['CS_SEXO'])
        
    if 'Situacao_Encerramento_Final' in df.columns:
        df['Situacao_Encerramento_Final'] = df['Situacao_Encerramento_Final'].map(mapa_encerramento).fillna(df['Situacao_Encerramento_Final'])
        
    if 'Tipo_Entrada' in df.columns:
        df['Tipo_Entrada'] = df['Tipo_Entrada'].map(mapa_entrada).fillna(df['Tipo_Entrada'])

print("✅ Formatação e Tipagem concluída!\n")

# ==============================================================================
# 4. UNICIDADE (REMOÇÃO DE DUPLICATAS)
# ==============================================================================
total_linhas_antes = len(df)
duplicatas_exatas = df.duplicated().sum()
print("--- ANÁLISE DE UNICIDADE ---")
print(f"Total de registos (pacientes) antes da limpeza: {total_linhas_antes}")
print(f"Linhas duplicadas exatas encontradas: {duplicatas_exatas}")

if duplicatas_exatas > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Nova volumetria após limpeza: {len(df)} registos únicos.")
else:
    print("A base não possui erros de duplicação exata!")

# Chave Sintética para achar duplicatas suspeitas
colunas_chave_sintetica = ['Idade_Anos', 'Sexo', 'Raca_Cor', 'Municipio_Residencia', 'Data_Diagnostico']
# Garante que usaremos os nomes corretos caso algo mude
colunas_chave_sintetica = [col for col in colunas_chave_sintetica if col in df.columns]

if colunas_chave_sintetica:
    df_suspeitos = df[df.duplicated(subset=colunas_chave_sintetica, keep=False)].copy()
    print(f"Total de casos suspeitos (Impressão Digital igual): {len(df_suspeitos)}")

# ==============================================================================
# 5. VALIDADE (INCONSISTÊNCIAS CRONOLÓGICAS)
# ==============================================================================
print("\n--- ANÁLISE DE VALIDADE ---")
if 'Data_Encerramento' in df.columns and 'Data_Diagnostico' in df.columns:
    mascara_erro_cronologico = df['Data_Encerramento'] < df['Data_Diagnostico']
    erros_cronologicos = df[mascara_erro_cronologico]

    df['Dias_Ate_Encerramento'] = (df['Data_Encerramento'] - df['Data_Diagnostico']).dt.days

    mascara_tempo_suspeito = (df['Dias_Ate_Encerramento'] < 0) | (df['Dias_Ate_Encerramento'] > 1825)
    erros_tempo = df[mascara_tempo_suspeito & df['Data_Encerramento'].notna()]

    print(f"Inconsistências Cronológicas (Encerramento antes do Diagnóstico): {len(erros_cronologicos)} casos")
    print(f"Tempos de Tratamento Suspeitos (> 5 anos ou negativos): {len(erros_tempo)} casos")

    # Limpeza Final
    volumetria_pre_filtro = len(df)
    df = df[~mascara_erro_cronologico & ~mascara_tempo_suspeito].copy()
    df.reset_index(drop=True, inplace=True)
    print(f"\nVolumetria após filtros de validade: {len(df)} pacientes prontos para análise.")
else:
    print("Colunas de data de diagnóstico ou encerramento não encontradas. Verificação pulada.")

# ==============================================================================
# 6. SALVAMENTO DO DATASET FINAL
# ==============================================================================
os.makedirs('/home/bruno/UESC/DataSUS/DadosTUB/data/processed', exist_ok=True)
caminho_dados_finais = '/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades_final.csv'

df.to_csv(caminho_dados_finais, index=False, encoding='utf-8')

print(f"\n✅ Base de dados nacional salva com sucesso em:\n{caminho_dados_finais}")
print("Você já pode abrir o Notebook 02 para recriar as Pirâmides Etárias!")

✅ Formatação e Tipagem concluída!

--- ANÁLISE DE UNICIDADE ---
Total de registos (pacientes) antes da limpeza: 251012
Linhas duplicadas exatas encontradas: 7
Nova volumetria após limpeza: 251005 registos únicos.
Total de casos suspeitos (Impressão Digital igual): 20598

--- ANÁLISE DE VALIDADE ---
Inconsistências Cronológicas (Encerramento antes do Diagnóstico): 426 casos
Tempos de Tratamento Suspeitos (> 5 anos ou negativos): 475 casos

Volumetria após filtros de validade: 250530 pacientes prontos para análise.

✅ Base de dados nacional salva com sucesso em:
/home/bruno/UESC/DataSUS/DadosTUB/data/processed/tuberculose_10_cidades_final.csv
Você já pode abrir o Notebook 02 para recriar as Pirâmides Etárias!
